# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

In [2]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [3]:
# Constants

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

In [4]:
# Create a messages list using the same format that we used for OpenAI

messages = [
    {"role": "user", "content": "Describe some of the business applications of Generative AI"}
]

In [5]:
payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }

In [7]:
response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
print(response.json()['message']['content'])

Generative AI has numerous business applications across various industries, including:

1. **Content Creation**: Generative AI can be used to create high-quality content such as images, videos, and text, reducing the need for human writers, designers, or artists.
2. **Marketing and Advertising**: Generative AI can help generate personalized product recommendations, social media posts, and ad copy, improving customer engagement and conversion rates.
3. **Customer Service**: Chatbots powered by generative AI can provide 24/7 customer support, answering frequently asked questions and routing complex issues to human representatives.
4. **Product Design**: Generative AI can be used to design products such as furniture, clothing, or electronics, reducing the time and cost associated with traditional design methods.
5. **Supply Chain Optimization**: Generative AI can analyze data from various sources to predict demand, optimize inventory levels, and identify potential bottlenecks in supply ch

# Introducing the ollama package

And now we'll do the same thing, but using the elegant ollama python package instead of a direct HTTP call.

Under the hood, it's making the same call as above to the ollama server running at localhost:11434

In [16]:
import ollama

response = ollama.chat(model=MODEL, messages=messages)
print(response['message']['content'])

Generative AI has numerous business applications across various industries, including:

1. **Content Creation**: Generative AI can be used to generate high-quality content such as articles, social media posts, and product descriptions, reducing the need for human writers and editors.
2. **Product Design**: Generative AI-powered design tools can create 3D models, prototypes, and even entire products, streamlining the product development process and saving costs.
3. **Image and Video Generation**: Generative AI can generate high-quality images and videos that can be used in marketing campaigns, advertising, and social media.
4. **Customer Service Chatbots**: Generative AI-powered chatbots can provide 24/7 customer support, answering common questions and routing complex issues to human representatives.
5. **Predictive Maintenance**: Generative AI can analyze sensor data from industrial equipment and predict when maintenance is required, reducing downtime and increasing overall efficiency.

# NOW the exercise for you

Take the code from day1 and incorporate it here, to build a website summarizer that uses Llama 3.2 running locally instead of OpenAI

In [8]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI



In [9]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

class Website:
    """
    A utility class to represent a Website that we have scraped
    """

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [10]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press coverage.
Connect
with me for

In [11]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [12]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [13]:
print(user_prompt_for(ed))

You are looking at a website titled Home - Edward Donner
The contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.

Home
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.

In [14]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [15]:
messages_for(ed)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'},
 {'role': 'user',
  'content': 'You are looking at a website titled Home - Edward Donner\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\nHome\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, posi

In [19]:
import ollama

response = ollama.chat(model=MODEL, messages=messages_for(ed))
display(Markdown(response['message']['content']))

**Summary**
================

### Website Overview

The website "Home - Edward Donner" is a personal blog or profile page of Edward Donner, a co-founder and CTO of Nebula.io. The website showcases his interests in AI, LLMs, music, and DJing.

### News/Announcements
------------------------

* **Mastering AI and LLM Engineering – Resources**: A collection of resources for mastering AI and LLM engineering.
* **From Software Engineer to AI Data Scientist – resources**: A set of resources for transitioning from a software engineer to an AI data scientist.
* **Outsmart LLM Arena – a battle of diplomacy and deviousness**: An introduction to the Outsmart LLM Arena, where LLMs compete against each other in a battle of diplomacy and deviousness. (Published June 26, 2024)
* **Choosing the Right LLM: Toolkit and Resources**: A collection of tools and resources for choosing the right LLM.
* **October 16, 2024 - Mastering AI and LLM Engineering – Resources**
* **August 6, 2024 - From Software Engineer to AI Data Scientist – resources**
* **November 13, 2024 - Mastering AI and LLM Engineering – Resources**

In [26]:
import openai
from typing import List

class OpenAIHelper:
    def __init__(self, api_key: str):
        openai.api_key = api_key

    def summarize_text(self, text: str) -> str:
        """
        Summarize a long piece of text using GPT
        """
        response = openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that summarizes text."},
                {"role": "user", "content": f"Please summarize this text concisely: {text}"}
            ],
            max_tokens=150,
            temperature=0.7
        )
        return response.choices[0].message.content

    def translate_text(self, text: str, target_language: str) -> str:
        """
        Translate text to the specified language
        """
        response = openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": f"You are a helpful translator. Translate to {target_language}."},
                {"role": "user", "content": text}
            ]
        )
        return response.choices[0].message.content

    def generate_social_media_posts(self, topic: str, num_posts: int = 3) -> List[str]:
        """
        Generate social media post ideas about a topic
        """
        response = openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "Generate engaging social media posts."},
                {"role": "user", "content": f"Create {num_posts} social media posts about {topic}"}
            ]
        )
        return response.choices[0].message.content.split("\n")

    def improve_writing(self, text: str) -> str:
        """
        Improve the writing style and grammar of a text
        """
        response = openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a professional editor."},
                {"role": "user", "content": f"Please improve this text while maintaining its meaning: {text}"}
            ]
        )
        return response.choices[0].message.content

# Example usage
if __name__ == "__main__":
    load_dotenv()
    api_key = os.getenv('OPENAI_API_KEY')
    
    # Check the key
    
    if not api_key:
        print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
    elif not api_key.startswith("sk-proj-"):
        print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
    elif api_key.strip() != api_key:
        print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
    else:
        print("API key found and looks good so far!")
    openaikey = os.getenv('OPENAI_API_KEY')
    ai = OpenAIHelper(openaikey)
    
    # Example 1: Summarize a long text
    long_text = """
    Machine learning is a subset of artificial intelligence that focuses on developing systems 
    that can learn from and make decisions based on data. It has numerous applications in 
    various fields, including healthcare, finance, and autonomous vehicles. The technology 
    continues to evolve and improve, leading to more sophisticated applications.
    """
    summary = ai.summarize_text(long_text)
    print("Summary:", summary)
    
    # Example 2: Translate text
    english_text = "Hello, how are you today?"
    spanish_translation = ai.translate_text(english_text, "Spanish")
    print("Translation:", spanish_translation)
    
    # Example 3: Generate social media posts
    posts = ai.generate_social_media_posts("artificial intelligence")
    print("\nSocial Media Posts:")
    for post in posts:
        print(f"- {post}")
    
    # Example 4: Improve writing
    draft = "I think that the movie was very good and had nice acting."
    improved = ai.improve_writing(draft)
    print("\nImproved Writing:", improved)

API key found and looks good so far!
Summary: Machine learning, a subset of artificial intelligence, develops systems that learn and make decisions from data, with applications in healthcare, finance, and autonomous vehicles, evolving towards more advanced use cases.
Translation: Hola, ¿cómo estás hoy?

Social Media Posts:
- 1. 🤖 "Did you know? Artificial intelligence is revolutionizing the way we live, work, and play. From virtual assistants to self-driving cars, the possibilities are endless! #AI #Technology"
- 
- 2. 🌟 "Harness the power of artificial intelligence to streamline your business operations and achieve greater efficiency. Discover how AI can transform your organization today! #ArtificialIntelligence #Innovation"
- 
- 3. 🧠 "Curious about the future of AI? Stay up-to-date with the latest advancements and breakthroughs in artificial intelligence. The future is now! #AI #TechTrends"

Improved Writing: I found the movie to be excellent, with enjoyable acting.
